<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Key insights:


In [1]:
!pip install datasets google-genai -q

In [2]:
import os
import io
import re
import time
import json
import numpy as np
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types
from datasets import load_dataset
from sklearn.metrics import mean_absolute_error

In [3]:
# ── API SETUP ────────────────────────────────────────────────
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)
print("Gemini client initialized.")

Gemini client initialized.


In [4]:
# ── LOAD DATASET ─────────────────────────────────────────────
print("Loading dataset...")
ds = load_dataset("naver-clova-ix/cord-v2")
print(f"Dataset loaded. Training records: {len(ds['train'])}")

Loading dataset...
Dataset loaded. Training records: 800


In [5]:
# ── BUILD df_receipt ─────────────────────────────────────────
receipt_data_df = []

for record in ds['train']:
    image            = record['image']
    ground_truth_str = record['ground_truth']
    total_price      = None

    try:
        ground_truth_json = json.loads(ground_truth_str)
        gt_parse          = ground_truth_json.get('gt_parse', {})
        total_info        = gt_parse.get('total', {})
        total_price_str   = total_info.get('total_price')

        if total_price_str is not None and isinstance(total_price_str, str):
            cleaned_price = total_price_str.replace(',', '.')
            parts         = re.findall(r'\d+', cleaned_price)
            if parts:
                last_dot_index = cleaned_price.rfind('.')
                if last_dot_index != -1 and last_dot_index > cleaned_price.rfind(parts[-1]):
                    integer_part  = ''.join(re.findall(r'\d', cleaned_price[:last_dot_index]))
                    decimal_part  = ''.join(re.findall(r'\d', cleaned_price[last_dot_index+1:]))
                    cleaned_price = f"{integer_part}.{decimal_part}"
                else:
                    cleaned_price = ''.join(parts)
            else:
                cleaned_price = ""

            if cleaned_price:
                try:
                    total_price = float(cleaned_price)
                except ValueError:
                    total_price = None

        receipt_data_df.append({'image': image, 'total_price': total_price})

    except Exception as e:
        continue

df_receipt = pd.DataFrame(receipt_data_df)
print(f"df_receipt created with {len(df_receipt)} records.")
print(f"Columns: {df_receipt.columns.tolist()}")
print(df_receipt[['total_price']].head())

df_receipt created with 800 records.
Columns: ['image', 'total_price']
   total_price
0    1591600.0
1     580965.0
2     334000.0
3     302016.0
4      48000.0


In [6]:
# ── STEP 1: RANDOMLY SELECT 15 RECORDS ───────────────────────
df_test_receipts = df_receipt.sample(n=15, random_state=42).reset_index(drop=True)
print(f"\nSelected {len(df_test_receipts)} records for testing.")
print(df_test_receipts[['total_price']].head())


Selected 15 records for testing.
   total_price
0      52500.0
1     105999.0
2      23600.0
3      60000.0
4      57900.0


In [7]:
# ── STEP 2: DEFINE PROMPT ────────────────────────────────────
prompt = "Extract the total amount from this receipt. Provide only the numerical value as a float, with no currency symbols, text, or extra characters."

In [8]:
# ── STEP 3: PROCESS WITH GEMINI ──────────────────────────────
predicted_prices = []

for i, row in df_test_receipts.iterrows():
    print(f"Processing receipt {i+1}/15...")
    try:
        img_byte_arr = io.BytesIO()
        row['image'].save(img_byte_arr, format='JPEG')
        img_bytes = img_byte_arr.getvalue()

        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                types.Part.from_bytes(data=img_bytes, mime_type='image/jpeg'),
                prompt
            ]
        )

        response_text   = response.text.strip()
        predicted_price = float(response_text)
        predicted_prices.append(predicted_price)
        print(f"  Predicted: {predicted_price}")

    except Exception as e:
        print(f"  Error: {e} — assigning NaN")
        predicted_prices.append(np.nan)

    time.sleep(5)

df_test_receipts['predicted_total_price'] = predicted_prices
print("\nPredictions complete!")

Processing receipt 1/15...
  Predicted: 52500.0
Processing receipt 2/15...
  Predicted: 105999.0
Processing receipt 3/15...
  Predicted: 23600.0
Processing receipt 4/15...
  Predicted: 60000.0
Processing receipt 5/15...
  Predicted: 57.9
Processing receipt 6/15...
  Predicted: 80000.0
Processing receipt 7/15...
  Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 36.636239668s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-

In [9]:
# ── STEP 4: EVALUATE ─────────────────────────────────────────
df_eval   = df_test_receipts.dropna(subset=['predicted_total_price', 'total_price'])
n_success = df_eval.shape[0]
mae       = mean_absolute_error(df_eval['total_price'], df_eval['predicted_total_price'])

def within_threshold(actual, predicted, threshold):
    return abs(actual - predicted) <= threshold * abs(actual)

df_eval = df_eval.copy()
df_eval['within_5pct']  = df_eval.apply(lambda r: within_threshold(r['total_price'], r['predicted_total_price'], 0.05), axis=1)
df_eval['within_10pct'] = df_eval.apply(lambda r: within_threshold(r['total_price'], r['predicted_total_price'], 0.10), axis=1)

acc_5pct  = df_eval['within_5pct'].mean()  * 100
acc_10pct = df_eval['within_10pct'].mean() * 100

print(f"\nSuccessful extractions : {n_success}/15")
print(f"Mean Absolute Error    : {mae:.2f}")
print(f"Accuracy within  5%    : {acc_5pct:.1f}%")
print(f"Accuracy within 10%    : {acc_10pct:.1f}%")


Successful extractions : 11/15
Mean Absolute Error    : 70202.45
Accuracy within  5%    : 72.7%
Accuracy within 10%    : 72.7%


In [10]:
# ── STEP 5: DISPLAY RESULTS ───────────────────────────────────
print("\nSample comparison (ground truth vs predicted):")
print(df_eval[['total_price', 'predicted_total_price', 'within_5pct', 'within_10pct']].head(10).to_string(index=False))



Sample comparison (ground truth vs predicted):
 total_price  predicted_total_price  within_5pct  within_10pct
     52500.0                52500.0         True          True
    105999.0               105999.0         True          True
     23600.0                23600.0         True          True
     60000.0                60000.0         True          True
     57900.0                   57.9        False         False
     80000.0                80000.0         True          True
    650100.0                  650.1        False         False
     14000.0                14000.0         True          True
    111000.0               111000.0         True          True
     65000.0                   65.0        False         False
